# A2CDF — Agentic AI-Driven Autonomous Cyber Defense Framework
### End-to-end research implementation notebook

This notebook implements the architecture described in the supplied work:
- Adaptive Context-Aware Threat Fusion (ACTF)
- Hierarchical Agentic Threat Reasoning Network (HATR-Net)
- Self-Evolving Autonomous Defense Engine (SADE)
- Explainable threat decisions
- Zero-day-style evaluation by withholding selected attack classes
- Metrics, confusion matrix, ROC/PR curves, response latency, and defense-policy analysis

**Safety note:** SADE is implemented in **simulation/safe mode**. It recommends/records actions rather than executing real network blocking, process termination, host isolation, or credential changes.


## 1. Implementation mapping

The supplied document describes CSE-CIC-IDS2018 as the primary dataset, CIC-IDS2017 for diversity/cross-environment evaluation, and MITRE ATT&CK STIX for threat-intelligence context. The framework normalizes heterogeneous security observations, withholds selected attack categories for unseen/zero-day evaluation, fuses contextual sources with attention, applies multiple reasoning agents plus temporal modeling, and uses a reward/cost-aware defense policy. fileciteturn0file0L157-L174

The notebook below provides a reproducible executable realization of those components. If the real datasets are not mounted, it automatically creates a **clearly labeled synthetic fallback** so every cell remains runnable.


In [ ]:

# ============================================================
# CELL 1 — Install / imports / configuration
# Purpose: Prepare a reproducible Python environment.
# ============================================================
import os, re, json, glob, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception:
    raise RuntimeError("PyTorch is required. In Colab run: !pip -q install torch")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, balanced_accuracy_score
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

# Update these paths if your benchmark datasets are already in Drive.
DATA_ROOTS = [
    "/content/drive/MyDrive",
    "/content",
    "/mnt/data"
]
OUTPUT_DIR = Path("/content/A2CDF_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:

# ============================================================
# CELL 2 — Dataset discovery
# Purpose: Find CIC-IDS2017 / CSE-CIC-IDS2018 CSV files.
# ============================================================
def discover_csvs(roots=DATA_ROOTS):
    hits = []
    for root in roots:
        if os.path.exists(root):
            for pat in ["*CIC*2017*.csv", "*CIC*2018*.csv",
                        "*IDS2017*.csv", "*IDS2018*.csv",
                        "*Friday*.csv", "*Monday*.csv", "*Tuesday*.csv",
                        "*Wednesday*.csv", "*Thursday*.csv"]:
                hits.extend(glob.glob(os.path.join(root, "**", pat), recursive=True))
    return sorted(set(hits))

csv_files = discover_csvs()
print("Candidate CSV files:", len(csv_files))
for p in csv_files[:20]:
    print(p)


In [ ]:

# ============================================================
# CELL 3 — Load and normalize a benchmark CSV
# Purpose: Convert heterogeneous flow records into a common table.
# ============================================================
LABEL_CANDIDATES = ["Label", "label", "Attack", "attack", "Class", "class"]

def clean_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[^A-Za-z0-9_]+", "_", str(c)).strip("_")
        for c in df.columns
    ]
    # remove duplicate column names
    seen, cols = {}, []
    for c in df.columns:
        if c not in seen:
            seen[c] = 0
            cols.append(c)
        else:
            seen[c] += 1
            cols.append(f"{c}_{seen[c]}")
    df.columns = cols
    return df

def find_label_col(df):
    for c in LABEL_CANDIDATES:
        if c in df.columns:
            return c
    for c in df.columns:
        if "label" in c.lower() or "attack" in c.lower():
            return c
    return None

def load_csv_safe(path, max_rows=120000):
    df = pd.read_csv(path, low_memory=False, nrows=max_rows)
    df = clean_columns(df)
    label = find_label_col(df)
    if label is None:
        raise ValueError(f"No attack-label column found in {path}")
    return df, label

# If a suitable CSV exists, load the first one.
REAL_DATA_USED = False
if csv_files:
    try:
        df, LABEL_COL = load_csv_safe(csv_files[0])
        REAL_DATA_USED = True
        print("Loaded:", csv_files[0])
        print("Shape:", df.shape, "| Label:", LABEL_COL)
    except Exception as e:
        print("Real dataset load failed:", e)
        df = None
else:
    df = None

if df is None:
    print("No compatible benchmark CSV found. Synthetic fallback will be generated in the next cell.")


In [ ]:

# ============================================================
# CELL 4 — Synthetic fallback
# Purpose: Keep the complete research pipeline executable when
# benchmark files are not mounted.
# ============================================================
def make_synthetic_cyber_dataset(n=30000, seed=42):
    rng = np.random.default_rng(seed)
    attacks = [
        "BENIGN", "DoS", "DDoS", "Brute Force",
        "Bot", "Web Attack", "Infiltration", "Heartbleed"
    ]
    probs = np.array([.46, .11, .10, .08, .08, .07, .06, .04])
    y = rng.choice(attacks, size=n, p=probs)

    attack_code = {a:i for i,a in enumerate(attacks)}
    a = np.array([attack_code[v] for v in y], dtype=float)

    X = pd.DataFrame({
        "Flow_Duration": np.exp(rng.normal(7.0 + .35*a, 1.0, n)),
        "Total_Fwd_Packets": rng.poisson(25 + 9*a, n) + 1,
        "Total_Bwd_Packets": rng.poisson(18 + 7*a, n) + 1,
        "Total_Length_Fwd": np.abs(rng.normal(900 + 450*a, 500, n)),
        "Total_Length_Bwd": np.abs(rng.normal(700 + 400*a, 450, n)),
        "Fwd_Packet_Length_Mean": np.abs(rng.normal(50 + 5*a, 25, n)),
        "Bwd_Packet_Length_Mean": np.abs(rng.normal(45 + 4*a, 23, n)),
        "Flow_Bytes_s": np.abs(rng.normal(18000 + 3500*a, 9000, n)),
        "Flow_Packets_s": np.abs(rng.normal(100 + 25*a, 45, n)),
        "Fwd_IAT_Mean": np.abs(rng.normal(900 - 35*a, 300, n)),
        "Bwd_IAT_Mean": np.abs(rng.normal(1000 - 30*a, 320, n)),
        "Active_Mean": np.abs(rng.normal(500 + 30*a, 180, n)),
        "Idle_Mean": np.abs(rng.normal(2500 + 60*a, 700, n)),
        "SYN_Flag_Count": rng.poisson(1 + .7*a, n),
        "RST_Flag_Count": rng.poisson(.5 + .35*a, n),
        "PSH_Flag_Count": rng.poisson(1 + .5*a, n),
        "ACK_Flag_Count": rng.poisson(5 + 1.5*a, n),
        "Packet_Length_Std": np.abs(rng.normal(30 + 6*a, 18, n)),
        "Subflow_Fwd_Bytes": np.abs(rng.normal(700 + 300*a, 300, n)),
        "Subflow_Bwd_Bytes": np.abs(rng.normal(600 + 280*a, 280, n)),
    })
    X["Protocol"] = rng.choice(["TCP", "UDP", "ICMP"], n, p=[.68,.27,.05])
    X["Service"] = rng.choice(["HTTP","HTTPS","DNS","SSH","FTP","Other"], n,
                               p=[.22,.20,.18,.10,.06,.24])
    X["Connection_State"] = rng.choice(["EST","SYN","FIN","RST","OTH"], n,
                                        p=[.58,.18,.10,.08,.06])
    X["Label"] = y
    return X

if df is None:
    df = make_synthetic_cyber_dataset()
    LABEL_COL = "Label"
    print("Synthetic dataset:", df.shape)


In [ ]:

# ============================================================
# CELL 5 — Preprocessing
# Purpose: Remove invalid/duplicate records, encode categorical
# fields, impute missing values, and scale numeric attributes.
# ============================================================
df = clean_columns(df)
LABEL_COL = find_label_col(df) or LABEL_COL

df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
df = df.dropna(subset=[LABEL_COL]).reset_index(drop=True)

# Keep a manageable sample for notebook execution.
MAX_ROWS = 50000
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)

y_text = df[LABEL_COL].astype(str).str.strip()
X_raw = df.drop(columns=[LABEL_COL])

# Remove obvious identifier/date columns where present.
drop_cols = []
for c in X_raw.columns:
    lc = c.lower()
    if any(k in lc for k in ["timestamp", "date", "time"]):
        drop_cols.append(c)
X_raw = X_raw.drop(columns=drop_cols, errors="ignore")

cat_cols = X_raw.select_dtypes(include=["object","category"]).columns.tolist()
num_cols = [c for c in X_raw.columns if c not in cat_cols]

for c in num_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")
    X_raw[c] = X_raw[c].fillna(X_raw[c].median())

for c in cat_cols:
    X_raw[c] = X_raw[c].astype(str).fillna("UNKNOWN")

X_enc = pd.get_dummies(X_raw, columns=cat_cols, dtype=float)
X_enc = X_enc.replace([np.inf,-np.inf], np.nan).fillna(0)

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_enc).astype(np.float32)

label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(y_text)
class_names = list(label_encoder.classes_)

# Binary target for anomaly/zero-day detection.
y_binary = (y_text.str.upper() != "BENIGN").astype(np.float32).values

print("Rows:", len(df))
print("Features after encoding:", X_scaled.shape[1])
print("Classes:", class_names)
print("Attack ratio:", round(float(y_binary.mean()), 4))


In [ ]:

# ============================================================
# CELL 6 — Mutual-information feature selection
# Purpose: Retain informative threat attributes as described by ACTF preprocessing.
# ============================================================
# MI is computed on a bounded sample for speed.
mi_n = min(10000, len(X_scaled))
mi_idx = np.random.default_rng(SEED).choice(len(X_scaled), mi_n, replace=False)

try:
    mi = mutual_info_classif(
        X_scaled[mi_idx], y_binary[mi_idx].astype(int),
        random_state=SEED
    )
except Exception:
    mi = np.var(X_scaled, axis=0)

TOP_K = min(64, X_scaled.shape[1])
selected_idx = np.argsort(mi)[-TOP_K:]
feature_names = np.array(X_enc.columns)[selected_idx]
X_sel = X_scaled[:, selected_idx].astype(np.float32)

print("Selected features:", len(feature_names))
print(list(feature_names[:20]))


In [ ]:

# ============================================================
# CELL 7 — Zero-day split
# Purpose: Hold out selected attack categories from training.
# ============================================================
attack_classes = [c for c in class_names if str(c).upper() != "BENIGN"]
# Select up to two attack families as unseen/zero-day classes.
zero_day_classes = attack_classes[-min(2, len(attack_classes)):] if attack_classes else []

is_zero_day = np.isin(y_text.values, zero_day_classes)
known_idx = np.where(~is_zero_day)[0]
zero_idx = np.where(is_zero_day)[0]

# Train/validation/test are created from known classes.
tr_idx, temp_idx = train_test_split(
    known_idx, test_size=0.30, random_state=SEED,
    stratify=y_multi[known_idx]
)
va_idx, te_known_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED,
    stratify=y_multi[temp_idx]
)

# Test includes known attacks plus completely withheld zero-day attacks.
test_idx = np.concatenate([te_known_idx, zero_idx])

print("Zero-day classes:", zero_day_classes)
print("Train:", len(tr_idx), "Val:", len(va_idx), "Test:", len(test_idx))
print("Zero-day test samples:", len(zero_idx))


In [ ]:

# ============================================================
# CELL 8 — PyTorch datasets
# Purpose: Create tensors for ACTF/HATR-Net.
# ============================================================
class CyberDataset(Dataset):
    def __init__(self, X, y_bin, y_multi):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.yb = torch.tensor(y_bin, dtype=torch.float32)
        self.ym = torch.tensor(y_multi, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yb[i], self.ym[i]

train_ds = CyberDataset(X_sel[tr_idx], y_binary[tr_idx], y_multi[tr_idx])
val_ds   = CyberDataset(X_sel[va_idx], y_binary[va_idx], y_multi[va_idx])
test_ds  = CyberDataset(X_sel[test_idx], y_binary[test_idx], y_multi[test_idx])

BATCH = 256
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=0)

# Source-specific views approximate network/endpoint/behavior/log/TI evidence.
INPUT_DIM = X_sel.shape[1]
N_SOURCES = 5
SOURCE_DIM = INPUT_DIM // N_SOURCES
PAD_DIM = N_SOURCES * SOURCE_DIM
if PAD_DIM < INPUT_DIM:
    SOURCE_DIM += 1
    PAD_DIM = N_SOURCES * SOURCE_DIM


## 2. ACTF + HATR-Net + SADE model

ACTF projects source-specific representations into a shared space and uses learned attention weights to produce a contextual threat representation, following the supplied description. fileciteturn0file0L208-L249

HATR-Net then uses specialized reasoning agents, combines their evidence, models temporal behavior with an LSTM, estimates severity, and predicts threat classes. fileciteturn0file0L257-L299


In [ ]:

# ============================================================
# CELL 9 — A2CDF neural architecture
# Purpose: Implement ACTF, multi-agent reasoning, temporal LSTM,
# binary anomaly head, multi-class threat head, and severity head.
# ============================================================
class ACTF(nn.Module):
    def __init__(self, in_dim, n_sources=5, d_model=128):
        super().__init__()
        self.n_sources = n_sources
        self.source_dim = int(np.ceil(in_dim / n_sources))
        padded = self.source_dim * n_sources
        self.proj = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.source_dim, d_model),
                nn.LayerNorm(d_model),
                nn.GELU()
            ) for _ in range(n_sources)
        ])
        self.score = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_model//2), nn.Tanh(),
                          nn.Linear(d_model//2, 1))
            for _ in range(n_sources)
        ])
        self.fusion = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(.15)
        )
        self.padded = padded

    def forward(self, x):
        if x.shape[1] < self.padded:
            x = F.pad(x, (0, self.padded-x.shape[1]))
        views = x.view(x.size(0), self.n_sources, self.source_dim)
        hs = torch.stack([self.proj[i](views[:,i]) for i in range(self.n_sources)], 1)
        scores = torch.cat([self.score[i](hs[:,i]) for i in range(self.n_sources)], 1)
        alpha = torch.softmax(scores, dim=1)
        fused = (alpha * hs).sum(dim=1)
        return self.fusion(fused), alpha.squeeze(-1)

class ReasoningAgent(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,d), nn.GELU(), nn.LayerNorm(d),
            nn.Dropout(.10), nn.Linear(d,d)
        )
    def forward(self,x): return self.net(x)

class A2CDF(nn.Module):
    def __init__(self, in_dim, n_classes, d=128, n_agents=4):
        super().__init__()
        self.actf = ACTF(in_dim, N_SOURCES, d)
        self.agents = nn.ModuleList([ReasoningAgent(d) for _ in range(n_agents)])
        self.lstm = nn.LSTM(d, d, batch_first=True)
        self.anomaly_head = nn.Sequential(nn.Linear(d,d//2),nn.GELU(),nn.Linear(d//2,1))
        self.class_head = nn.Sequential(nn.Linear(d,d//2),nn.GELU(),nn.Linear(d//2,n_classes))
        self.severity_head = nn.Sequential(nn.Linear(d,d//2),nn.GELU(),nn.Linear(d//2,1),nn.Sigmoid())

    def forward(self,x):
        z, alpha = self.actf(x)
        agent_z = torch.stack([a(z) for a in self.agents], 1).mean(1)
        # A short sequence is formed from current and agent evidence.
        seq = torch.stack([z, agent_z, z + agent_z], dim=1)
        temporal,_ = self.lstm(seq)
        h = temporal[:,-1]
        anomaly = self.anomaly_head(h).squeeze(-1)
        threat = self.class_head(h)
        severity = self.severity_head(h).squeeze(-1)
        return anomaly, threat, severity, alpha

model = A2CDF(INPUT_DIM, len(class_names)).to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:

# ============================================================
# CELL 10 — Training
# Purpose: Train the complete detection/reasoning network.
# ============================================================
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, n = 0.0, 0
    for xb, yb, ym in loader:
        xb, yb, ym = xb.to(DEVICE), yb.to(DEVICE), ym.to(DEVICE)
        with torch.set_grad_enabled(train):
            ab, tm, sv, _ = model(xb)
            # Multi-task objective:
            # anomaly detection + known-class classification + severity.
            loss_a = F.binary_cross_entropy_with_logits(ab, yb)
            # For zero-day design, known training data still has standard classes.
            loss_c = F.cross_entropy(tm, ym)
            target_sev = torch.clamp(.25 + .70*yb, 0, 1)
            loss_s = F.mse_loss(sv, target_sev)
            loss = loss_a + loss_c + .25*loss_s
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                optimizer.step()
        total_loss += float(loss.item()) * len(xb)
        n += len(xb)
    return total_loss/max(n,1)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
EPOCHS = 8
history = {"train":[], "val":[]}
best = np.inf

for ep in range(1, EPOCHS+1):
    tr_loss = run_epoch(train_loader, True)
    va_loss = run_epoch(val_loader, False)
    history["train"].append(tr_loss); history["val"].append(va_loss)
    if va_loss < best:
        best = va_loss
        torch.save(model.state_dict(), OUTPUT_DIR/"best_a2cdf.pt")
    print(f"Epoch {ep:02d}/{EPOCHS} | train={tr_loss:.4f} | val={va_loss:.4f}")

model.load_state_dict(torch.load(OUTPUT_DIR/"best_a2cdf.pt", map_location=DEVICE))


In [ ]:

# ============================================================
# CELL 11 — Curves
# Purpose: Plot training/validation loss.
# ============================================================
plt.figure(figsize=(7,4))
plt.plot(history["train"], marker="o", label="Train")
plt.plot(history["val"], marker="o", label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("A2CDF Training Curve")
plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"training_curve.png", dpi=300)
plt.show()


In [ ]:

# ============================================================
# CELL 12 — Evaluation utilities
# Purpose: Compute standard detection and zero-day metrics.
# ============================================================
@torch.no_grad()
def predict_loader(loader):
    model.eval()
    all_a, all_t, all_s, all_yb, all_ym, all_alpha = [],[],[],[],[],[]
    for xb,yb,ym in loader:
        xb = xb.to(DEVICE)
        a,t,s,alpha = model(xb)
        all_a.append(torch.sigmoid(a).cpu().numpy())
        all_t.append(torch.softmax(t,1).cpu().numpy())
        all_s.append(s.cpu().numpy())
        all_yb.append(yb.numpy()); all_ym.append(ym.numpy())
        all_alpha.append(alpha.cpu().numpy())
    return (np.concatenate(all_a), np.concatenate(all_t),
            np.concatenate(all_s), np.concatenate(all_yb),
            np.concatenate(all_ym), np.concatenate(all_alpha))

p_a, p_t, p_s, yb_test, ym_test, alphas = predict_loader(test_loader)

pred_bin = (p_a >= .50).astype(int)
pred_multi = p_t.argmax(1)

metrics = {
    "Accuracy (%)": accuracy_score(yb_test, pred_bin)*100,
    "Precision (%)": precision_score(yb_test, pred_bin, zero_division=0)*100,
    "Recall (%)": recall_score(yb_test, pred_bin, zero_division=0)*100,
    "F1-Score (%)": f1_score(yb_test, pred_bin, zero_division=0)*100,
    "Balanced Accuracy (%)": balanced_accuracy_score(yb_test, pred_bin)*100,
    "AUC-ROC (%)": roc_auc_score(yb_test, p_a)*100 if len(np.unique(yb_test))>1 else np.nan,
    "AUC-PR (%)": average_precision_score(yb_test, p_a)*100 if len(np.unique(yb_test))>1 else np.nan
}
print(pd.Series(metrics).round(3))

print("\nMulti-class report:")
print(classification_report(
    ym_test, pred_multi, labels=np.arange(len(class_names)),
    target_names=class_names, zero_division=0
))


In [ ]:

# ============================================================
# CELL 13 — Zero-day evaluation
# Purpose: Measure behavior on attack families withheld from training.
# ============================================================
zero_test_mask = np.isin(y_text.values[test_idx], zero_day_classes)
if zero_test_mask.any():
    zy = yb_test[zero_test_mask]
    zp = p_a[zero_test_mask]
    zpred = (zp >= .50).astype(int)
    print("Zero-day classes:", zero_day_classes)
    print("Zero-day samples:", len(zy))
    print("Zero-day detection accuracy:", round(accuracy_score(zy,zpred)*100,3))
    print("Zero-day recall:", round(recall_score(zy,zpred,zero_division=0)*100,3))
    print("Zero-day F1:", round(f1_score(zy,zpred,zero_division=0)*100,3))
else:
    print("No zero-day samples available.")


In [ ]:

# ============================================================
# CELL 14 — Confusion matrix
# Purpose: Visualize binary threat detection.
# ============================================================
cm = confusion_matrix(yb_test, pred_bin, labels=[0,1])
plt.figure(figsize=(5,4))
plt.imshow(cm, interpolation="nearest")
plt.title("A2CDF Threat Detection Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks([0,1],["Benign","Attack"]); plt.yticks([0,1],["Benign","Attack"])
for i in range(2):
    for j in range(2):
        plt.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"confusion_matrix.png", dpi=300)
plt.show()


## 3. SADE — Self-Evolving Autonomous Defense Engine

The supplied work defines available responses such as blocking traffic, terminating connections, isolating hosts, restricting credentials/processes/services, or monitoring; it also uses reward, long-term value, operational cost, and a severity threshold. fileciteturn0file0L304-L348

This notebook implements those ideas as a **simulation policy**. It does not execute real defensive changes on a machine or network.


In [ ]:

# ============================================================
# CELL 15 — Safe SADE environment
# Purpose: Simulate response selection, reward, policy adaptation,
# and operational cost without touching a real system.
# ============================================================
ACTIONS = [
    "MONITOR",
    "BLOCK_TRAFFIC",
    "TERMINATE_CONNECTION",
    "ISOLATE_HOST",
    "RESTRICT_CREDENTIAL",
    "TERMINATE_PROCESS",
    "RESTRICT_SERVICE"
]

ACTION_COST = np.array([0.05, 0.20, 0.30, 0.65, 0.50, 0.60, 0.55], dtype=np.float32)

class SADE:
    def __init__(self, n_states=5, n_actions=len(ACTIONS), lr=.08,
                 gamma=.90, severity_threshold=.60):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        self.threshold = severity_threshold
        self.Q = np.zeros((n_states, n_actions), dtype=np.float32)

    def state(self, anomaly, severity):
        # 0 benign/low ... 4 critical
        score = .6*anomaly + .4*severity
        return int(np.clip(np.floor(score*5), 0, 4))

    def choose(self, state, epsilon=.08):
        if np.random.rand() < epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.Q[state]))

    def reward(self, anomaly, severity, action):
        # Simulation reward: containment benefit minus operational cost.
        expected = 0.0
        if severity >= self.threshold:
            if action == 0: expected -= 1.0
            else: expected += min(1.5, severity*1.5)
        else:
            if action == 0: expected += .6
            else: expected -= .4
        expected -= .35*ACTION_COST[action]
        return expected

    def update(self, s, a, r, s2):
        target = r + self.gamma*np.max(self.Q[s2])
        self.Q[s,a] += self.lr*(target-self.Q[s,a])

sade = SADE()
sade_log = []

# Train policy in simulation using model predictions.
for ep in range(30):
    order = np.random.permutation(len(p_a))
    ep_return = 0
    for i in order[:min(2000,len(order))]:
        anomaly, severity = float(p_a[i]), float(p_s[i])
        s = sade.state(anomaly, severity)

        # Enforce threshold: low-severity events remain in monitoring.
        if severity < sade.threshold:
            a = 0
        else:
            a = sade.choose(s, epsilon=max(.02, .20*(1-ep/30)))

        r = sade.reward(anomaly, severity, a)
        s2 = sade.state(anomaly, min(1.0, severity + .05*r))
        sade.update(s,a,r,s2)
        ep_return += r

    sade_log.append(ep_return)

plt.figure(figsize=(7,4))
plt.plot(sade_log)
plt.xlabel("Policy episode"); plt.ylabel("Cumulative simulated reward")
plt.title("SADE Policy Adaptation")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"sade_reward.png", dpi=300)
plt.show()

print("Learned Q-table:")
print(pd.DataFrame(sade.Q, columns=ACTIONS).round(3))


In [ ]:

# ============================================================
# CELL 16 — Generate safe defense decisions
# Purpose: Produce an auditable incident-response table.
# ============================================================
def explain_event(x, anomaly, severity, alpha):
    feature_strength = np.abs(x) * (np.mean(alpha) if alpha.ndim else 1.0)
    top = np.argsort(feature_strength)[-5:][::-1]
    return [(str(feature_names[j]), float(feature_strength[j])) for j in top]

records = []
for i in range(min(1000, len(p_a))):
    anomaly, severity = float(p_a[i]), float(p_s[i])
    s = sade.state(anomaly, severity)
    a = 0 if severity < sade.threshold else int(np.argmax(sade.Q[s]))
    evidence = explain_event(X_sel[test_idx[i]], anomaly, severity, alphas[i])

    records.append({
        "Sample": i+1,
        "Threat_Probability": anomaly,
        "Severity": severity,
        "Threat_Detected": int(anomaly >= .5),
        "Defense_Action_Simulated": ACTIONS[a],
        "Evidence": "; ".join([f"{k}={v:.3f}" for k,v in evidence])
    })

decision_df = pd.DataFrame(records)
decision_df.to_csv(OUTPUT_DIR/"SADE_decision_log.csv", index=False)
decision_df.head(10)


## 4. Explainability

The paper specifies feature contribution analysis and ranking by absolute contribution, then links the explanation to the selected defense action. fileciteturn0file0L356-L391

The next cell provides a lightweight, model-agnostic perturbation explanation that works without requiring a separate SHAP installation.


In [ ]:

# ============================================================
# CELL 17 — Perturbation explainability
# Purpose: Estimate which selected features most affect the threat score.
# ============================================================
@torch.no_grad()
def threat_score(x_np):
    x = torch.tensor(x_np[None,:], dtype=torch.float32, device=DEVICE)
    a,_,_,_ = model(x)
    return float(torch.sigmoid(a).cpu().item())

def local_feature_importance(x, top_k=15):
    base = threat_score(x)
    vals = []
    for j in range(len(x)):
        xx = x.copy()
        xx[j] = 0.0
        delta = base - threat_score(xx)
        vals.append(delta)
    vals = np.asarray(vals)
    idx = np.argsort(np.abs(vals))[-top_k:][::-1]
    return pd.DataFrame({
        "Feature": feature_names[idx],
        "Contribution": vals[idx],
        "Absolute_Contribution": np.abs(vals[idx])
    })

example_imp = local_feature_importance(X_sel[test_idx[0]])
example_imp


In [ ]:

# ============================================================
# CELL 18 — ROC and PR curves
# Purpose: Plot threshold-independent detection performance.
# ============================================================
from sklearn.metrics import roc_curve, precision_recall_curve

if len(np.unique(yb_test)) > 1:
    fpr,tpr,_ = roc_curve(yb_test,p_a)
    plt.figure(figsize=(6,4))
    plt.plot(fpr,tpr,label=f"AUC={metrics['AUC-ROC (%)']/100:.3f}")
    plt.plot([0,1],[0,1],"--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("A2CDF ROC Curve"); plt.legend(); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"roc_curve.png",dpi=300)
    plt.show()

    pr,rc,_ = precision_recall_curve(yb_test,p_a)
    plt.figure(figsize=(6,4))
    plt.plot(rc,pr,label=f"AP={metrics['AUC-PR (%)']/100:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("A2CDF Precision-Recall Curve"); plt.legend(); plt.tight_layout()
    plt.savefig(OUTPUT_DIR/"pr_curve.png",dpi=300)
    plt.show()


In [ ]:

# ============================================================
# CELL 19 — Source attention analysis
# Purpose: Show how ACTF weights the five contextual sources.
# ============================================================
source_names = ["Network Traffic","Endpoint Telemetry","User Behavior",
                "System Logs","Threat Intelligence"]
mean_alpha = alphas.mean(axis=0)

plt.figure(figsize=(7,4))
plt.bar(source_names, mean_alpha)
plt.ylabel("Mean ACTF Attention")
plt.title("Adaptive Context-Aware Threat Fusion")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"actf_attention.png", dpi=300)
plt.show()

print(pd.DataFrame({"Source":source_names, "Mean_Attention":mean_alpha}).round(4))


In [ ]:

# ============================================================
# CELL 20 — Response latency benchmark
# Purpose: Measure model inference latency and simulated policy latency.
# ============================================================
model.eval()
latencies = []
with torch.no_grad():
    for xb,_,_ in test_loader:
        xb = xb.to(DEVICE)
        if DEVICE == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(xb)
        if DEVICE == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter()-t0)/len(xb)*1000)

latency_ms = float(np.mean(latencies))
print(f"Mean model inference latency: {latency_ms:.4f} ms/sample")

summary = pd.DataFrame([{
    **metrics,
    "Mean Inference Latency (ms/sample)": latency_ms,
    "Trainable Parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "Zero-Day Classes": ", ".join(map(str,zero_day_classes))
}])
summary.to_csv(OUTPUT_DIR/"A2CDF_overall_metrics.csv", index=False)
summary.round(4)


In [ ]:

# ============================================================
# CELL 21 — Save all research outputs
# Purpose: Create a single reproducibility bundle.
# ============================================================
import shutil

np.save(OUTPUT_DIR/"ACTF_attention.npy", alphas)
pd.DataFrame({
    "Feature": feature_names,
    "Mutual_Information": mi[selected_idx]
}).sort_values("Mutual_Information", ascending=False).to_csv(
    OUTPUT_DIR/"selected_features_MI.csv", index=False
)

with open(OUTPUT_DIR/"configuration.json","w") as f:
    json.dump({
        "seed": SEED,
        "device": DEVICE,
        "real_dataset_used": REAL_DATA_USED,
        "zero_day_classes": [str(x) for x in zero_day_classes],
        "n_features": int(INPUT_DIM),
        "epochs": EPOCHS,
        "batch_size": BATCH,
        "architecture": "ACTF + multi-agent HATR-Net + LSTM + SADE simulation"
    }, f, indent=2)

zip_path = shutil.make_archive("/content/A2CDF_Results", "zip", OUTPUT_DIR)
print("Results folder:", OUTPUT_DIR)
print("ZIP:", zip_path)


## 5. Important reproducibility notes

1. **For the actual paper experiment**, mount/download the stated CSE-CIC-IDS2018 and CIC-IDS2017 datasets and point `DATA_ROOTS` to their folders. The supplied document explicitly names those datasets and MITRE ATT&CK STIX as sources. fileciteturn0file0L157-L174
2. The synthetic fallback is only for pipeline validation; it should not be presented as benchmark evidence.
3. The zero-day experiment withholds selected classes from training and evaluates them only at test time, matching the document's stated evaluation design. fileciteturn0file0L169-L174
4. SADE is intentionally simulation-only. Real defensive execution should require analyst approval, authentication, authorization, logging, and environment-specific safety controls.
5. The notebook produces the standard detection metrics plus zero-day results, ACTF attention, explainability, SADE policy rewards, decision logs, and latency.
